# Local Mother Runner — native Windows / NPO D1

This notebook orchestrates the **existing** local Mother runner. It installs the training environment, checks manually supplied inputs, runs the full NPO D1 pipeline, and seals outputs. It does not change the loss, models, precision, optimizer, schedule, evaluation or memory strategy. GPU memory capacity is **not a launch gate**; actual setup/runtime failures are preserved.

## Before opening a run

Use native Windows, Python **3.11 or 3.12**, Git and an NVIDIA driver. The notebook kernel only controls the run; training runs in `.venv-local\Scripts\python.exe`, created by Step 4. No kernel switch is needed. Jupyter itself must already be installed to open a notebook.

**Run a working copy, not this tracked template.** Jupyter saves outputs into its notebook; the unchanged D1 runtime rejects a dirty Git checkout. From the repository root in PowerShell:

```powershell
py -3.12 -m pip install notebook
New-Item -ItemType Directory -Force results/local/notebooks | Out-Null
$notebook = "results/local/notebooks/LOCAL_MOTHER_RUNNER-$(Get-Date -Format 'yyyyMMdd-HHmmss').ipynb"
Copy-Item LOCAL_MOTHER_RUNNER.ipynb $notebook
py -3.12 -m notebook $notebook
```

Open that working copy, select its Python 3.11/3.12 kernel, edit **Step 1 only**, and run cells in order. The working copy is ignored by Git; do not edit/save the root template during a run. Input details: [models](../../../models/NPO/README.md), [data](../../../data/NPO/README.md), [full guide](../../../handbook/LOCAL_MOTHER_RUNNER.md) (links relative to the working-copy location).

Each attempt gets a new `results/local/<run-id>`. Never rerun a completed stage to retry an old attempt. After failure, inspect Step 7, fix the cause, restart the kernel, and run a new attempt. Resume requires an explicitly selected identity-verified native checkpoint; partial logs are not a checkpoint.

## Step 1 — Configure locations and GPU

`REPO_ROOT=None` discovers the repository above the notebook's working directory. Set an absolute path if your editor starts the kernel elsewhere. `MODELS=None` and `DATA=None` use `models/NPO` and `data/NPO` under that root. GPU `0` is the physical CUDA device selection passed to the child; it becomes logical device zero there. No model/data download is performed. The target, retain-only oracle, tokenizer, generation config and all six TOFU files are required.

Use raw strings for Windows paths, for example `Path(r"D:\fq-ssiuu")`. Do not put passwords or tokens in this notebook.

In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace

REPO_ROOT = None  # Example: Path(r"C:\MY-FYP\fq-ssiuu")
MODELS = None  # Example: Path(r"D:\models\NPO")
DATA = None  # Example: Path(r"D:\data\NPO")
GPU = 0
RESUME_CHECKPOINT = None  # Example: Path(r"D:\saved-run\training_checkpoint")

if REPO_ROOT is None:
    REPO_ROOT = next(
        (
            p
            for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/Local/mother_runner.py").is_file()
        ),
        None,
    )
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT to your fq-ssiuu checkout in Step 1.")
REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "notebooks/Local/mother_runner.py").is_file():
    raise FileNotFoundError(f"Local runner missing in {REPO_ROOT}")
if not isinstance(GPU, int) or isinstance(GPU, bool) or GPU < 0:
    raise ValueError("GPU must be a nonnegative integer.")
args = SimpleNamespace(
    models=Path(MODELS).expanduser().resolve() if MODELS else REPO_ROOT / "models/NPO",
    data=Path(DATA).expanduser().resolve() if DATA else REPO_ROOT / "data/NPO",
    gpu=GPU,
    resume_checkpoint=Path(RESUME_CHECKPOINT).expanduser().resolve() if RESUME_CHECKPOINT else None,
)
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)
print("Notebook kernel:", sys.executable)
print("Inputs / GPU:", vars(args))

## Step 2 — Start an immutable attempt and record source identity

Creates the output directory, checks that Git is clean, records the commit/tree and recipe hashes, and checks native Windows/Python compatibility. It records the hardware selection without a GPU-memory capacity gate. If this cell reports a dirty checkout, preserve your changes and use the ignored working copy described above. Do not bypass the source checks.

Timestamped stage messages also go to `raw/steps.jsonl`; `raw/active_step.json` records the latest stage. Setup/training commands additionally record `raw/active_command.json` with the command, PID and status.

In [ ]:
from notebooks.Local.mother_runner import LocalRun

previous_run = globals().get("run")
if previous_run is not None and not previous_run.finished:
    raise RuntimeError("An attempt is already active. Finish it before starting another.")
run = LocalRun(args)
print("RUN DIRECTORY:", run.run)
run.initialize()

## Step 3 — Verify manual model and dataset files

Calls the existing `experiments/reproductions/NPO/local_inputs.py` verifier. Missing or hash-mismatched files stop here, before dependency installation/training. Exact input receipts are written to `raw/input_files.json`. Large model hashes can take time; this is disk reading, not training.

In [ ]:
run.verify_inputs()
print("Verified input receipt:", run.raw / "input_files.json")

## Step 4 — Create the training environment and install dependencies

Uses the existing setup order: `.venv-local`, build tools, CUDA PyTorch, pinned dependencies, hash-pinned DeepSpeed, editable project install, `pip check`, and package inventory. Internet access is needed for package installation. Model/data access remains offline.

Every command prints START/COMPLETE/FAILED, its exact arguments, and live stdout/stderr here. Separate byte-preserving logs and execution receipts remain on disk, e.g. `dependencies.stdout.log`, `dependencies.stderr.log`, `dependencies.execution.json`. A failed installation stops dependent steps; no alternative optimizer or backend is silently substituted. The notebook kernel does not need Torch or DeepSpeed installed.

In [ ]:
run.setup_environment()
print("Training interpreter:", run.python)
print("Installed versions:", run.raw / "inventory.stdout.log")

## Step 5 — Execute the unchanged full NPO D1 pipeline

Launches `experiments/reproductions/NPO/train_runtime.py` with the same recipe and local input arguments as the PowerShell runner. It performs:

1. Runtime/version, input and adapter validation, then the tiny GPU optimizer/resume check.
2. Retain-only oracle evaluation, target model/engine and full frozen reference setup.
3. The existing 62-update schedule, scheduled evaluations and optional explicit checkpoint resume.
4. Native-state saving, final evaluation/model export, parameter-change checks and final-model reload validation.

The entire runtime stays in **one child process** to preserve its state and sequence. Live stdout/stderr appears below. Read-only polling also reports existing evidence files: `training/forward_observations.jsonl` marks a completed forward **before** backward; `training/microbatches.jsonl` reports completed microbatches. Displayed JSONL summaries show the latest available record, not every record. Full records stay in those files. An artifact-observed message is not proof that its stage passed. During quiet operations a 30-second heartbeat confirms the child is still running, not that it is making progress.

The child's `NPO/traceback.log` is displayed when present, including source file and line. Import/startup failures instead appear in `npo.stderr.log`. An OOM is a real failed run; this notebook does not change memory handling.

Use **Interrupt Kernel** to cancel normally: the owned process tree is stopped before partial outputs are sealed. Closing the browser alone does not stop the kernel. A forced kernel kill, PC crash or power loss cannot guarantee cleanup or sealing: check `active_step.json`, `active_command.json`, raw logs and the OS process list before starting another attempt.

In [ ]:
run.train()

## Step 6 — Seal outputs and report the result

Success is accepted only when the existing child emits its required acceptance status. This cell writes `run_result.json` and seals the raw artifacts with the existing archive utility. Caught failures in earlier stages already do this automatically before raising a notebook error. `finish()` is idempotent, so it is safe to use after such a failure.

Sealing may require substantial time and extra disk space for a second copy of checkpoint files. Archive failure is reported separately in `archive_status.json`, preserves raw files and gives a nonzero exit code. Do not modify raw files after sealing. If you intentionally stop between cells, calling this cell seals an incomplete attempt as CANCELLED.

In [ ]:
exit_code = run.finish()
if exit_code:
    raise RuntimeError(f"Attempt did not complete successfully; inspect Step 7 and {run.run}")

## Step 7 — Inspect a result or diagnose a failed/crashed attempt

This cell is read-only and can be run after an earlier cell fails. After a kernel restart, execute Step 1 and set `INSPECT_RUN` to the old run directory; skip Steps 2–6. It prints the current stage/command, final result, archive status and available tracebacks. `INSPECT_RUN=None` selects this session's run. No automatic 'latest run' or checkpoint selection is performed.

Start with the failing command's stderr and the NPO traceback. For backward OOMs, compare the last forward observation with the last completed microbatch. Do not interpret a finite forward loss as a completed update or evidence of forgetting.

In [ ]:
INSPECT_RUN = None  # Example: REPO_ROOT / "results/local/local-YYYYMMDD-HHMMSS-id"

if INSPECT_RUN is None:
    if "run" not in globals():
        raise ValueError("Set INSPECT_RUN to the exact attempt directory.")
    inspect_run = run.run
else:
    inspect_run = Path(INSPECT_RUN).expanduser().resolve()
if not inspect_run.is_dir():
    raise FileNotFoundError(inspect_run)
for relative in (
    "raw/active_step.json",
    "raw/active_command.json",
    "raw/run_result.json",
    "archive_status.json",
    "raw/NPO/RESULT.json",
    "raw/traceback.log",
    "raw/NPO/traceback.log",
):
    path = inspect_run / relative
    print(f"\n--- {relative} ---")
    print(path.read_text(encoding="utf-8", errors="replace") if path.is_file() else "Not written.")
print("\nCommand logs:")
for path in sorted((inspect_run / "raw").glob("*.log")):
    print(path)
print("NPO evidence:", inspect_run / "raw/NPO")